# Kafka Silver — Validation

## Validation Purpose

This notebook independently validates the Kafka Silver dataset produced by the transformation notebook.

The validation verifies:

- Record preservation
- Event ID uniqueness
- User and video identifiers
- Timestamp parsing
- Temporal derived fields
- Interaction normalization
- Device normalization
- Watch-time preservation
- Data-quality flags
- Preservation of intentionally dirty records
- Silver output schema
- Silver output partitioning

The validation does not modify the Silver dataset.

## Validation Principles

- Validate the transformed output independently from the transformation logic.
- Do not remove or modify records during validation.
- Verify that source events are preserved at the original event-level grain.
- Verify that data-quality flags correctly identify detected issues.
- Verify that intentionally dirty values remain available for analysis.
- Verify that technically normalized fields have the expected data types.
- Do not treat intentional dirty data as a validation failure when it is correctly preserved and flagged.

In [ ]:
if 'spark' in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/08 00:34:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
KAFKA_BRONZE_PATH = (
    "s3a://end-to-end-streaming-data-platform-bronze/"
    "kafka/ingestion_data=2026-09-01-Sep/"
)

KAFKA_SILVER_PATH = (
    "s3a://end-to-end-streaming-data-platform-silver/"
    "kafka/events_silver.parquet"
)

#df_kafka_bronze = spark.read.parquet(KAFKA_BRONZE_PATH)
df_kafka_silver = spark.read.parquet(KAFKA_SILVER_PATH)

print("Kafka Bronze loaded.")
print("Kafka Silver loaded.")

26/09/08 00:36:19 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Kafka Bronze loaded.
Kafka Silver loaded.


In [5]:
#bronze_count = df_kafka_bronze.count()
silver_count = df_kafka_silver.count()

print("Bronze records:", bronze_count)
print("Silver records:", silver_count)
print("Records preserved:", bronze_count == silver_count)

Bronze records: 16074
Silver records: 16074
Records preserved: True


In [6]:
duplicate_event_ids = (
    df_kafka_silver
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_event_ids = (
    df_kafka_silver
    .filter(F.col("event_id").isNull())
    .count()
)

print("Duplicate event_id groups:", duplicate_event_ids)
print("Null event_id:", null_event_ids)

Duplicate event_id groups: 0
Null event_id: 0


In [7]:
required_columns = [
    "event_id",
    "event_timestamp",
    "event_timestamp_raw",
    "user_id",
    "video_id",
    "interaction_type",
    "watch_time_sec",
    "device_type",
    "event_date",
    "event_hour",
    "ingestion_date",
    "batch_id",
]

missing_columns = [
    c for c in required_columns
    if c not in df_kafka_silver.columns
]

print("Missing required columns:", missing_columns)

Missing required columns: []


In [8]:
df_kafka_silver.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- interaction_type: string (nullable = true)
 |-- watch_time_sec: integer (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_timestamp_raw: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- dq_missing_user_id: boolean (nullable = true)
 |-- dq_missing_video_id: boolean (nullable = true)
 |-- dq_missing_watch_time: boolean (nullable = true)
 |-- dq_unexpected_watch_time: boolean (nullable = true)
 |-- dq_invalid_watch_time: boolean (nullable = true)
 |-- dq_invalid_interaction: boolean (nullable = true)
 |-- dq_invalid_device: boolean (nullable = true)
 |-- dq_invalid_timestamp: boolean (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- batch_id: string (nullable = true)



In [9]:
invalid_timestamp_count = (
    df_kafka_silver
    .filter(F.col("dq_invalid_timestamp"))
    .count()
)

valid_timestamp_count = (
    df_kafka_silver
    .filter(
        F.col("event_timestamp").isNotNull()
        & ~F.col("dq_invalid_timestamp")
    )
    .count()
)

print("Invalid timestamp:", invalid_timestamp_count)
print("Valid timestamps:", valid_timestamp_count)

Invalid timestamp: 1
Valid timestamps: 16073


In [10]:
df_kafka_silver.filter(
    F.col("dq_invalid_timestamp")
).select(
    "event_id",
    "event_timestamp_raw",
    "event_timestamp",
    "dq_invalid_timestamp"
).show(truncate=False)

+------------------------------------+-------------------+---------------+--------------------+
|event_id                            |event_timestamp_raw|event_timestamp|dq_invalid_timestamp|
+------------------------------------+-------------------+---------------+--------------------+
|01a05ead-c2d0-7d8f-bb25-2479f26bca74|not-a-timestamp    |NULL           |true                |
+------------------------------------+-------------------+---------------+--------------------+



In [11]:
temporal_mismatch = (
    df_kafka_silver
    .filter(
        (
            F.col("event_timestamp").isNotNull()
            & (
                F.col("event_date") !=
                F.to_date(F.col("event_timestamp"))
            )
        )
        |
        (
            F.col("event_timestamp").isNotNull()
            & (
                F.col("event_hour") !=
                F.hour(F.col("event_timestamp"))
            )
        )
    )
    .count()
)

print("Temporal derivation mismatches:", temporal_mismatch)

Temporal derivation mismatches: 0


In [12]:
invalid_temporal_values = (
    df_kafka_silver
    .filter(F.col("dq_invalid_timestamp"))
    .filter(
        F.col("event_date").isNotNull()
        | F.col("event_hour").isNotNull()
    )
    .count()
)

print("Invalid timestamps with derived temporal values:",
      invalid_temporal_values)

Invalid timestamps with derived temporal values: 0


In [13]:
VALID_INTERACTIONS = (
    "like",
    "play",
    "complete",
    "pause",
)

invalid_interaction_flag_errors = (
    df_kafka_silver
    .filter(
        (
            F.col("interaction_type").isin(*VALID_INTERACTIONS)
            & F.col("dq_invalid_interaction")
        )
        |
        (
            ~F.col("interaction_type").isin(*VALID_INTERACTIONS)
            & ~F.col("dq_invalid_interaction")
        )
    )
    .count()
)

print(
    "Interaction flag errors:",
    invalid_interaction_flag_errors
)

Interaction flag errors: 0


In [14]:
df_kafka_silver.filter(
    F.col("interaction_type") == "share"
).select(
    "event_id",
    "interaction_type",
    "dq_invalid_interaction"
).show()

+--------------------+----------------+----------------------+
|            event_id|interaction_type|dq_invalid_interaction|
+--------------------+----------------+----------------------+
|01a05ead-c2ce-7ea...|           share|                  true|
+--------------------+----------------+----------------------+



In [15]:
VALID_DEVICES = (
    "mobile",
    "tv",
    "web",
)

invalid_device_flag_errors = (
    df_kafka_silver
    .filter(
        (
            F.col("device_type").isin(*VALID_DEVICES)
            & F.col("dq_invalid_device")
        )
        |
        (
            ~F.col("device_type").isin(*VALID_DEVICES)
            & ~F.col("dq_invalid_device")
        )
    )
    .count()
)

print("Device flag errors:", invalid_device_flag_errors)

Device flag errors: 0


In [16]:
df_kafka_silver.filter(
    F.col("device_type") == "console"
).select(
    "event_id",
    "device_type",
    "dq_invalid_device"
).show()

+--------------------+-----------+-----------------+
|            event_id|device_type|dq_invalid_device|
+--------------------+-----------+-----------------+
|01a05ead-c2cf-775...|    console|             true|
+--------------------+-----------+-----------------+



In [17]:
bronze_watch_time = (
    df_kafka_bronze
    .select("event_id", "watch_time_sec")
    .withColumnRenamed(
        "watch_time_sec",
        "bronze_watch_time"
    )
)

silver_watch_time = (
    df_kafka_silver
    .select("event_id", "watch_time_sec")
)

watch_time_mismatches = (
    bronze_watch_time
    .join(silver_watch_time, "event_id")
    .filter(
        ~(
            F.col("bronze_watch_time").eqNullSafe(
                F.col("watch_time_sec")
            )
        )
    )
    .count()
)

print("Watch-time mismatches:", watch_time_mismatches)

Watch-time mismatches: 0


In [18]:
watch_time_flag_errors = (
    df_kafka_silver
    .filter(
        (
            (
                F.col("watch_time_sec").isNotNull()
                & (
                    (F.col("watch_time_sec") <= 0)
                    | (F.col("watch_time_sec") > 120)
                )
            )
            != F.col("dq_invalid_watch_time")
        )
    )
    .count()
)

print("Invalid watch-time flag errors:", watch_time_flag_errors)

Invalid watch-time flag errors: 0


In [19]:
df_kafka_silver.filter(
    F.col("dq_invalid_watch_time")
).select(
    "event_id",
    "watch_time_sec",
    "dq_invalid_watch_time"
).show(10, truncate=False)

+------------------------------------+--------------+---------------------+
|event_id                            |watch_time_sec|dq_invalid_watch_time|
+------------------------------------+--------------+---------------------+
|01a05ead-c2cc-73c4-9b8b-ce2a6f9c1ce5|0             |true                 |
|01a05ead-c2cd-7fdd-bda1-d9747623c5fa|121           |true                 |
|01a05ead-c2cf-775c-8d21-022d972f0bec|1429          |true                 |
|01a05ead-c2d3-7924-9e42-550f5a169556|6149          |true                 |
|01a05ead-c2d4-72a2-86ea-63a616d64a60|2270          |true                 |
|01a05ead-c2e1-732d-99f8-dd8e27e4ff15|4964          |true                 |
|01a05ead-c2e9-70d0-ae44-cc2d56376451|1907          |true                 |
|01a05ead-c2eb-7507-8a4b-af9f675be403|2833          |true                 |
|01a05ead-c2f2-7243-8774-e1539dd69b68|1687          |true                 |
|01a05ead-c2f4-7c1d-9bef-f9189fc36941|3120          |true                 |
+-----------

In [20]:
missing_watch_flag_errors = (
    df_kafka_silver
    .filter(
        (
            (
                F.col("interaction_type").isin("play", "complete")
                & F.col("watch_time_sec").isNull()
            )
            != F.col("dq_missing_watch_time")
        )
    )
    .count()
)

unexpected_watch_flag_errors = (
    df_kafka_silver
    .filter(
        (
            (
                F.col("interaction_type").isin("like", "pause")
                & F.col("watch_time_sec").isNotNull()
            )
            != F.col("dq_unexpected_watch_time")
        )
    )
    .count()
)

print("Missing watch-time flag errors:",
      missing_watch_flag_errors)

print("Unexpected watch-time flag errors:",
      unexpected_watch_flag_errors)

Missing watch-time flag errors: 0
Unexpected watch-time flag errors: 0


In [22]:
bronze_ids = (
    df_kafka_bronze
    .select(
        "event_id",
        F.col("user_id").alias("bronze_user_id"),
        F.col("video_id").alias("bronze_video_id")
    )
)

silver_ids = (
    df_kafka_silver
    .select(
        "event_id",
        F.col("user_id").alias("silver_user_id"),
        F.col("video_id").alias("silver_video_id")
    )
)

id_mismatches = (
    bronze_ids
    .join(silver_ids, "event_id")
    .filter(
        ~F.col("bronze_user_id").eqNullSafe(
            F.col("silver_user_id")
        )
        |
        ~F.col("bronze_video_id").eqNullSafe(
            F.col("silver_video_id")
        )
    )
    .count()
)

print("User/video ID mismatches:", id_mismatches)

User/video ID mismatches: 0


In [23]:
dq_columns = [
    "dq_missing_user_id",
    "dq_missing_video_id",
    "dq_missing_watch_time",
    "dq_unexpected_watch_time",
    "dq_invalid_watch_time",
    "dq_invalid_interaction",
    "dq_invalid_device",
    "dq_invalid_timestamp",
]

for col_name in dq_columns:
    null_count = (
        df_kafka_silver
        .filter(F.col(col_name).isNull())
        .count()
    )

    print(f"{col_name}: {null_count} NULLs")

dq_missing_user_id: 0 NULLs
dq_missing_video_id: 0 NULLs
dq_missing_watch_time: 0 NULLs
dq_unexpected_watch_time: 0 NULLs
dq_invalid_watch_time: 0 NULLs
dq_invalid_interaction: 0 NULLs
dq_invalid_device: 0 NULLs
dq_invalid_timestamp: 0 NULLs


In [24]:
grain_violations = (
    df_kafka_silver
    .groupBy("event_id")
    .count()
    .filter(F.col("count") != 1)
    .count()
)

print("Grain violations:", grain_violations)

Grain violations: 0


In [25]:
print("Null ingestion_date:",
      df_kafka_silver.filter(
          F.col("ingestion_date").isNull()
      ).count())

print("Null batch_id:",
      df_kafka_silver.filter(
          F.col("batch_id").isNull()
      ).count())

print("Distinct batch_id:",
      df_kafka_silver.select("batch_id").distinct().count())

Null ingestion_date: 0
Null batch_id: 0
Distinct batch_id: 1


In [26]:
dirty_records_preserved = {
    "share_interaction": (
        df_kafka_silver
        .filter(F.col("interaction_type") == "share")
        .count()
    ),

    "console_device": (
        df_kafka_silver
        .filter(F.col("device_type") == "console")
        .count()
    ),

    "watch_time_over_120": (
        df_kafka_silver
        .filter(F.col("watch_time_sec") > 120)
        .count()
    ),

    "zero_watch_time": (
        df_kafka_silver
        .filter(F.col("watch_time_sec") == 0)
        .count()
    ),

    "invalid_timestamp": (
        df_kafka_silver
        .filter(F.col("dq_invalid_timestamp"))
        .count()
    ),
}

for name, count in dirty_records_preserved.items():
    print(f"{name}: {count}")

share_interaction: 1
console_device: 1
watch_time_over_120: 4084
zero_watch_time: 1
invalid_timestamp: 1


In [ ]:
validation_results = {
    "record_count_preserved": bronze_count == silver_count,
    "duplicate_event_ids": duplicate_event_ids == 0,
    "null_event_ids": null_event_ids == 0,
    "required_columns_present": len(missing_columns) == 0,
    "timestamp_flag_correct": invalid_timestamp_count == 1,
    "temporal_derivations_correct": temporal_mismatch == 0,
    "invalid_temporal_values": invalid_temporal_values == 0,
    "interaction_flags_correct": invalid_interaction_flag_errors == 0,
    "device_flags_correct": invalid_device_flag_errors == 0,
    "watch_time_preserved": watch_time_mismatches == 0,
    "watch_time_flags_correct": watch_time_flag_errors == 0,
    "missing_watch_flags_correct": missing_watch_flag_errors == 0,
    "unexpected_watch_flags_correct": unexpected_watch_flag_errors == 0,
    "ids_preserved": id_mismatches == 0,
    "dq_flags_not_null": all(
        df_kafka_silver.filter(
            F.col(c).isNull()
        ).count() == 0
        for c in dq_columns
    ),
    "original_grain_preserved": grain_violations == 0,
}

print("=" * 70)
print("KAFKA SILVER — VALIDATION SUMMARY")
print("=" * 70)

for test_name, result in validation_results.items():
    print(f"{test_name}: {'PASS' if result else 'FAIL'}")

print("=" * 70)

overall_result = all(validation_results.values())

print(
    "OVERALL:",
    "PASS" if overall_result else "FAIL"
)

In [3]:
df_kafka_silver.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- interaction_type: string (nullable = true)
 |-- watch_time_sec: integer (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_timestamp_raw: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- dq_missing_user_id: boolean (nullable = true)
 |-- dq_missing_video_id: boolean (nullable = true)
 |-- dq_missing_watch_time: boolean (nullable = true)
 |-- dq_unexpected_watch_time: boolean (nullable = true)
 |-- dq_invalid_watch_time: boolean (nullable = true)
 |-- dq_invalid_interaction: boolean (nullable = true)
 |-- dq_invalid_device: boolean (nullable = true)
 |-- dq_invalid_timestamp: boolean (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- batch_id: string (nullable = true)



In [4]:
gold_fact_events = (
    df_kafka_silver
    .filter(
        (F.col("dq_missing_user_id") == False) &
        (F.col("dq_missing_video_id") == False) &
        (F.col("dq_missing_watch_time") == False) &
        (F.col("dq_unexpected_watch_time") == False) &
        (F.col("dq_invalid_watch_time") == False) &
        (F.col("dq_invalid_interaction") == False) &
        (F.col("dq_invalid_device") == False) &
        (F.col("dq_invalid_timestamp") == False)
    )
    .select(
        F.col("event_id"),
        F.col("event_timestamp"),
        F.col("user_id"),
        F.col("video_id"),
        F.col("interaction_type"),
        F.col("watch_time_sec"),
        F.col("device_type")
    )
)

In [5]:
gold_fact_events \
    .write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-gold/fact_events/")

In [6]:
gold_fact_events.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- interaction_type: string (nullable = true)
 |-- watch_time_sec: integer (nullable = true)
 |-- device_type: string (nullable = true)

